# Introvert vs Extrovert Classification

**Goal**: Predict whether a person is an introvert or extrovert using the provided personality dataset.

**Author**: Wickramanayaka Mudalige Ashan Maleesha Madhushan  
**Contact**: +94 740288813 | wmashan2@gmail.com
**Assignment**: AI/ML Intern — Round 02 Technical Assignment (Decryptogen)

---

## What this notebook does

1. **Load & inspect data** (missing values, quick preview).
2. **Preprocess features** (impute missing values, scale numeric, one-hot encode categorical).
3. **Train baselines** (Logistic Regression, Random Forest).
4. **Cross-validate** to estimate generalization.
5. **Tune Random Forest** with randomized search for better accuracy.
6. **Evaluate** on a holdout test set.
7. **Save artifacts** for deployment.
8. **Demo prediction** on a sample input.

---

## Outputs to look for

- Cross-validation F1 (macro) scores
- Final test accuracy + classification report
- Saved model file: `models/personality_model.joblib`
- Saved metadata: `models/metadata.json`

## Imports

Load libraries used for data prep, modeling, and evaluation.

In [33]:
# Load core libraries for data prep, modeling, and persistence
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV,
 )
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

## Load Data

Read the CSV and quickly inspect the rows and missing values.

In [34]:
# Load dataset and preview a few rows
data_path = Path('personality_dataset.csv')
df = pd.read_csv(data_path)
df.head()

,Time_spent_Alone,Stage_fear,Social_event_attendance,Going_outside,Drained_after_socializing,Friends_circle_size,Post_frequency,Personality
0,4.0,No,4.0,6.0,No,13.0,5.0,Extrovert
1,9.0,Yes,0.0,0.0,Yes,0.0,3.0,Introvert
2,9.0,Yes,1.0,2.0,Yes,5.0,2.0,Introvert
3,0.0,No,6.0,7.0,No,14.0,8.0,Extrovert
4,3.0,No,9.0,4.0,No,8.0,5.0,Extrovert


In [21]:
# Quick missing-value check
df.isna().sum()

Time_spent_Alone             63
Stage_fear                   73
Social_event_attendance      62
Going_outside                66
Drained_after_socializing    52
Friends_circle_size          77
Post_frequency               65
Personality                   0
dtype: int64

## Preprocessing

Define numeric/categorical pipelines and split train/test.

In [35]:
# Define features, preprocessing pipelines, and train/test split
target_col = 'Personality'
X = df.drop(columns=[target_col])
y = df[target_col]

categorical_features = ['Stage_fear', 'Drained_after_socializing']
numeric_features = [col for col in X.columns if col not in categorical_features]

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Train Models

Fit baseline models and compare metrics.

In [36]:
# Train baseline models and compare performance
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    clf = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    f1 = f1_score(y_test, preds, pos_label='Introvert')
    acc = accuracy_score(y_test, preds)
    print(f'[{name}] Accuracy: {acc:.3f} | F1 (Introvert): {f1:.3f}')
    return clf, acc, f1

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
rf_model = RandomForestClassifier(
    n_estimators=300, random_state=42, class_weight='balanced'
 )

lr_clf, lr_acc, lr_f1 = evaluate_model('Logistic Regression', lr_model, X_train, X_test, y_train, y_test)
rf_clf, rf_acc, rf_f1 = evaluate_model('Random Forest', rf_model, X_train, X_test, y_train, y_test)

[Logistic Regression] Accuracy: 0.917 | F1 (Introvert): 0.917
[Random Forest] Accuracy: 0.902 | F1 (Introvert): 0.900


## Cross-Validation and Tuning

Use cross-validation to estimate generalization and tune the Random Forest.

In [37]:
# Cross-validation for a more stable accuracy estimate
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv = cross_val_score(lr_clf, X, y, cv=cv, scoring="f1_macro")
rf_cv = cross_val_score(rf_clf, X, y, cv=cv, scoring="f1_macro")
print(f"LogReg CV F1 (macro): {lr_cv.mean():.3f} +/- {lr_cv.std():.3f}")
print(f"RF CV F1 (macro): {rf_cv.mean():.3f} +/- {rf_cv.std():.3f}")

# Randomized search: better accuracy with less compute than full grid
rf_param_dist = {
    "model__n_estimators": [200, 300, 400, 500, 700],
    "model__max_depth": [None, 6, 10, 14, 18],
    "model__min_samples_split": [2, 4, 8, 12],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 0.7],
}

rf_base = Pipeline(steps=[("preprocess", preprocess), ("model", RandomForestClassifier(
    random_state=42, class_weight="balanced"
))])

rf_search = RandomizedSearchCV(
    rf_base,
    param_distributions=rf_param_dist,
    n_iter=20,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42
)
rf_search.fit(X_train, y_train)

tuned_rf_clf = rf_search.best_estimator_
tuned_rf_score = rf_search.best_score_
print("Best RF params:", rf_search.best_params_)
print(f"Best RF CV F1 (macro): {tuned_rf_score:.3f}")

LogReg CV F1 (macro): 0.923 +/- 0.009
RF CV F1 (macro): 0.920 +/- 0.006
Best RF params: {'model__n_estimators': 200, 'model__min_samples_split': 8, 'model__min_samples_leaf': 1, 'model__max_features': 'log2', 'model__max_depth': None}
Best RF CV F1 (macro): 0.939


## Final Evaluation

Evaluate the tuned model on the holdout set.

In [38]:
# Evaluate the tuned model on the holdout set
best_clf = tuned_rf_clf
best_name = "Random Forest (tuned)"

print(f"Using best model: {best_name}")
preds = best_clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))
print("Confusion matrix:")
print(confusion_matrix(y_test, preds))
print("Classification report:")
print(classification_report(y_test, preds))

Using best model: Random Forest (tuned)
Accuracy: 0.9155172413793103
Confusion matrix:
[[266  32]
 [ 17 265]]
Classification report:
              precision    recall  f1-score   support

   Extrovert       0.94      0.89      0.92       298
   Introvert       0.89      0.94      0.92       282

    accuracy                           0.92       580
   macro avg       0.92      0.92      0.92       580
weighted avg       0.92      0.92      0.92       580



## Save Artifacts

Persist the model and metadata for deployment.

In [40]:
# Persist model and metadata for deployment
models_dir = Path('models')
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / 'personality_model.joblib'
metadata_path = models_dir / 'metadata.json'

joblib.dump(best_clf, model_path)

metadata = {
    'model_name': best_name,
    'features': X.columns.tolist(),
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'classes': sorted(y.unique().tolist())
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

model_path, metadata_path

(PosixPath('models/personality_model.joblib'),
 PosixPath('models/metadata.json'))

## Prediction Demo

Load saved artifacts and run a single sample.

In [42]:
# Demo: load saved model and run one sample prediction
import json
from pathlib import Path

import joblib
import pandas as pd

models_dir = Path("models")
model_path = models_dir / "personality_model.joblib"
metadata_path = models_dir / "metadata.json"

model = joblib.load(model_path)
with metadata_path.open("r", encoding="utf-8") as f:
    meta = json.load(f)

sample = {
    "Time_spent_Alone": 0,
    "Stage_fear": "No",
    "Social_event_attendance": 7.0,
    "Going_outside": 6.0,
    "Drained_after_socializing": "No",
    "Friends_circle_size": 10.0,
    "Post_frequency": 4.0,
}

df = pd.DataFrame([{k: sample[k] for k in meta["features"]}])
proba = model.predict_proba(df)
pred = model.classes_[proba.argmax(axis=1)][0]

print("Prediction:", pred)
print("Confidence:", float(proba.max()))

Prediction: Extrovert
Confidence: 0.9615009476417018
